# UncertaintyDecode — Quick Start Notebook

**Uncertainty-Guided KV Cache Eviction for Hallucination-Aware LLM Inference**

This notebook runs the full UncertaintyDecode pipeline on a free T4 GPU.

What we'll do:
1. Install dependencies
2. Test the Triton kernel (uncertainty head overhead)
3. Demonstrate the eviction policy on synthetic data
4. Run a mini TruthfulQA eval comparing LRU vs UncertaintyDecode
5. Visualize uncertainty scores

**Runtime: ~25 minutes on T4**

> Change Runtime → T4 GPU or A100 GPU before running.

## Step 1: Install Dependencies

In [ ]:
# Install vLLM and other dependencies
!pip install vllm==0.8.5 triton>=2.2.0 datasets scikit-learn -q
!pip install git+https://github.com/Arul-sathya/uncertainty-decode.git -q
print('Installation complete!')

In [ ]:
import torch
import numpy as np
import json

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Step 2: Triton Kernel Benchmark

In [ ]:
from uncertainty_decode.kernels.dirichlet_kernel import benchmark_triton_vs_pytorch

print('Benchmarking Triton kernel vs PyTorch...')
print('(Testing on Llama-3.1-8B dimensions: D=4096, proj=256)\n')

configs = [
    (1, 128),   # single request, short
    (4, 512),   # small batch
    (8, 512),   # production batch
    (8, 2048),  # long context
]

print(f"{'Config':<20} {'PyTorch (ms)':<15} {'Triton (ms)':<15} {'Speedup':<10} {'Overhead %'}")
print('-'*70)

# Estimate full forward pass time for a typical LLM decode step (~50ms on T4)
full_forward_ms = 50.0

for B, T in configs:
    results = benchmark_triton_vs_pytorch(B=B, T=T, D=4096, proj_size=256, K=2)
    overhead_pct = results['triton_ms'] / full_forward_ms * 100
    print(f"B={B}, T={T:<5}         {results['pytorch_ms']:<15.2f} {results['triton_ms']:<15.2f} {results['speedup']:<10.1f} {overhead_pct:.1f}%")

print('\n✓ Target: < 2ms overhead (< 4% of forward pass on T4)')

## Step 3: Eviction Policy Demonstration

In [ ]:
from uncertainty_decode.eviction.policy import UncertaintyEvictionPolicy
from uncertainty_decode.eviction.block_scorer import BlockScorer

# Simulate a 128-token sequence with uncertainty scores
# Pattern: tokens 32-64 are highly uncertain (imagine a factual claim being generated)
T = 128
uncertainty_scores = torch.rand(T) * 0.3  # baseline: low uncertainty
uncertainty_scores[32:64] = torch.rand(32) * 0.3 + 0.65  # spike: uncertain region
uncertainty_scores[48:56] = torch.rand(8) * 0.1 + 0.85   # peak: very uncertain

print('Uncertainty score statistics:')
print(f'  Mean: {uncertainty_scores.mean():.3f}')
print(f'  Max:  {uncertainty_scores.max():.3f}')
print(f'  Pct above threshold (0.65): {(uncertainty_scores > 0.65).float().mean():.1%}')

# Visualize block-level uncertainty
scorer = BlockScorer(block_size=8, aggregation='max', threshold=0.65)
print('\n' + scorer.visualize_ascii(uncertainty_scores))

# Show which blocks would be evicted by LRU vs UncertaintyDecode
block_scores = scorer.score_blocks(uncertainty_scores)
keep_ids, evict_ids = scorer.compute_eviction_budget(block_scores, kv_budget=0.6)

print(f'\nWith 60% KV budget (keep {len(keep_ids)}/16 blocks):')
print(f'  UncertaintyDecode keeps: blocks {sorted(keep_ids)}')
print(f'  UncertaintyDecode evicts: blocks {sorted(evict_ids)}')

# LRU would evict oldest blocks (0, 1, 2, 3, 4, 5 = first 6 blocks)
lru_evict = list(range(6))  # oldest = blocks 0-5
print(f'\n  LRU would evict: blocks {lru_evict}')

# Check: does LRU accidentally evict uncertain blocks?
lru_evicts_uncertain = any(block_scores[i].is_protected for i in lru_evict)
ud_evicts_uncertain = any(block_scores[i].is_protected for i in evict_ids)
print(f'\n  LRU evicts uncertain blocks: {lru_evicts_uncertain}')
print(f'  UncertaintyDecode evicts uncertain blocks: {ud_evicts_uncertain}')

## Step 4: Mini TruthfulQA Evaluation (50 samples)

In [ ]:
# NOTE: This cell requires Hugging Face access to Llama-3.1-8B.
# If you don't have access, uncomment the Mistral-7B line instead.

# MODEL = 'meta-llama/Meta-Llama-3.1-8B-Instruct'  # needs HF token
MODEL = 'mistralai/Mistral-7B-Instruct-v0.2'        # open access

import subprocess
import sys

print(f'Running mini TruthfulQA eval on {MODEL}')
print('(50 samples, comparing LRU vs UncertaintyDecode)\n')

result = subprocess.run([
    sys.executable, 'evals/eval_truthfulqa.py',
    '--model', MODEL,
    '--n-samples', '50',
    '--policies', 'lru', 'uncertainty_decode',
    '--kv-budget', '0.6',
    '--output', 'results/colab_truthfulqa.json',
], capture_output=True, text=True)

print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])

In [ ]:
# Load and display results
import json
from pathlib import Path

results_path = Path('results/colab_truthfulqa.json')
if results_path.exists():
    with open(results_path) as f:
        results = json.load(f)

    print('\n=== RESULTS ===')
    print(f'{"Policy":<22} {"MC1 Acc":<12} {"Halluc Rate":<15} {"Unc AUROC"}')
    print('-'*60)
    for r in results:
        print(f"{r['policy']:<22} {r['mc1_accuracy']:<12.3f} "
              f"{r['hallucination_rate']:<15.3f} {r['uncertainty_auroc']:.3f}")

    lru = next(r for r in results if r['policy'] == 'lru')
    ud = next((r for r in results if r['policy'] == 'uncertainty_decode'), None)
    if ud:
        delta = (ud['mc1_accuracy'] - lru['mc1_accuracy']) / lru['mc1_accuracy'] * 100
        print(f'\nUncertaintyDecode improvement: {delta:+.1f}% over LRU')
else:
    print('Results not found — did the eval run complete?')

## Step 5: Uncertainty Score Visualization

In [ ]:
# Visualize uncertainty scores across a generated sequence
# Shows WHERE in the generation the model becomes uncertain
import matplotlib
matplotlib.use('Agg')  # headless
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from uncertainty_decode.eviction.uncertainty_head import DirichletEvidenceHead, UncertaintyConfig
from uncertainty_decode.eviction.block_scorer import BlockScorer

# Generate synthetic uncertainty pattern representing a factual QA response
# (In real use, these come from the model's hidden states)
torch.manual_seed(42)
T = 80

# Simulate: confident during question parsing, uncertain during answer generation
u = torch.rand(T) * 0.2 + 0.1       # base: low uncertainty (confident context)
u[10:20] = torch.rand(10) * 0.2 + 0.6  # reasoning phase: moderate uncertainty
u[20:35] = torch.rand(15) * 0.2 + 0.7  # factual claim: HIGH uncertainty → PROTECT
u[35:50] = torch.rand(15) * 0.15 + 0.5 # elaboration: moderate
u[55:65] = torch.rand(10) * 0.2 + 0.68 # another uncertain claim → PROTECT

scorer = BlockScorer(block_size=8, aggregation='max', threshold=0.65)
block_scores = scorer.score_blocks(u)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), gridspec_kw={'height_ratios': [3, 1]})

# Plot 1: Token-level uncertainty
colors = ['#e74c3c' if v > 0.65 else '#3498db' for v in u.numpy()]
ax1.bar(range(T), u.numpy(), color=colors, alpha=0.8, width=0.9)
ax1.axhline(y=0.65, color='black', linestyle='--', linewidth=1.5, label='Threshold (θ=0.65)')
ax1.fill_between(range(T), 0.65, u.numpy(), where=(u.numpy() > 0.65),
                  color='#e74c3c', alpha=0.3, label='Protected zone')
ax1.set_ylabel('Epistemic Uncertainty', fontsize=11)
ax1.set_title('UncertaintyDecode: Per-Token Uncertainty → KV Block Protection', fontsize=12)
ax1.legend(loc='upper right')
ax1.set_ylim(0, 1.05)
ax1.set_xlim(-1, T)

# Plot 2: Block-level protection map
block_colors = ['#e74c3c' if b.is_protected else '#2ecc71' for b in block_scores]
block_starts = [b.token_start for b in block_scores]
block_widths = [b.token_end - b.token_start for b in block_scores]
ax2.barh(0, block_widths, left=block_starts, height=0.6, color=block_colors, alpha=0.85)
for b in block_scores:
    mid = (b.token_start + b.token_end) / 2
    label = '🔒' if b.is_protected else '✓'
    ax2.text(mid, 0, label, ha='center', va='center', fontsize=9)

protected_patch = mpatches.Patch(color='#e74c3c', alpha=0.85, label='Protected (high uncertainty)')
evict_patch = mpatches.Patch(color='#2ecc71', alpha=0.85, label='Evictable (low uncertainty)')
ax2.legend(handles=[protected_patch, evict_patch], loc='upper right', fontsize=9)
ax2.set_xlabel('Token position', fontsize=11)
ax2.set_ylabel('KV Blocks', fontsize=11)
ax2.set_xlim(-1, T)
ax2.set_yticks([])

plt.tight_layout()
plt.savefig('results/uncertainty_visualization.png', dpi=150, bbox_inches='tight')
print('Figure saved to results/uncertainty_visualization.png')

n_protected = sum(1 for b in block_scores if b.is_protected)
print(f'\n{n_protected}/{len(block_scores)} blocks protected ({n_protected/len(block_scores):.0%})')
print(f'KV memory saved by evicting non-protected blocks: {1 - n_protected/len(block_scores):.0%}')

In [ ]:
# Display the figure inline
from IPython.display import Image
Image('results/uncertainty_visualization.png')

## Summary

You've now:
1. ✅ Verified the Triton kernel runs in < 2ms
2. ✅ Seen how the eviction policy protects uncertain token windows
3. ✅ Compared hallucination rates on TruthfulQA
4. ✅ Visualized uncertainty scores across a generated sequence

**Next steps:**
- Run the full eval: `eval_longbench.py` (needs ~4h on T4)
- Train the uncertainty head: `scripts/train_uncertainty_head.py` (~30min on A100)
- Submit to arXiv using `docs/paper_outline.md`

**GitHub:** https://github.com/Arul-sathya/uncertainty-decode
